In [1]:
# Importing Pre Requisites

import requests
import pandas as pd
import time

# Defining API Endpoint
URL = "https://scanner.tradingview.com/india/scan"

# Defining Headers
HEADERS = {"User-Agent": "Mozilla/5.0", "Content-Type": "application/json"}

# Defining Columns

COLUMNS = [
    "name",
    "description",
    "close",
    "change",
    "volume",
    "relative_volume_10d_calc",
    "market_cap_basic",
    "price_earnings_ttm",
    "price_book_fq",
    "return_on_equity",
    "return_on_assets",
    "earnings_per_share_basic_ttm",
    "earnings_per_share_diluted_yoy_growth_ttm",
    "dividend_yield_recent",
    "RSI",
    "SMA20",
    "SMA50",
    "SMA200",
    "sector",
]


# Defining Structure

TOTAL_ROWS = 5000  # Rows Count
CHUNK_SIZE = 1000  # Batch Size

all_data = []  # Storing our Data

session = requests.Session()  # Defining Session


# Starting Extraction Loop
for start in range(0, TOTAL_ROWS, CHUNK_SIZE):

    end = start + CHUNK_SIZE

    payload = {
        "filter": [],
        "options": {"lang": "en"},
        "symbols": {"query": {"types": []}, "tickers": []},
        "columns": COLUMNS,
        "sort": {"sortBy": "market_cap_basic", "sortOrder": "desc"},
        "range": [start, end],
    }

    try:

        response = session.post(URL, json=payload, headers=HEADERS, timeout=30)

        if response.status_code != 200:
            print("Request failed")
            break

        data = response.json()

        rows = data.get("data", [])

        if not rows:
            break

        for item in rows:

            try:

                exchange_info = item.get("s", "")

                # STRICT NSE FILTER
                if not exchange_info.startswith("NSE:"):
                    continue

                d = item.get("d", [])

                if len(d) < 19:
                    continue

                row = {
                    "Exchange": "NSE",
                    "Symbol": d[0],
                    "Company": d[1],
                    "Price": d[2],
                    "Change %": d[3],
                    "Volume": d[4],
                    "Relative Volume": d[5],
                    "Market Cap": d[6],
                    "P/E": d[7],
                    "P/B": d[8],
                    "ROE": d[9],
                    "ROA": d[10],
                    "EPS TTM": d[11],
                    "EPS Growth YoY": d[12],
                    "Dividend Yield": d[13],
                    "RSI": d[14],
                    "SMA20": d[15],
                    "SMA50": d[16],
                    "SMA200": d[17],
                    "Sector": d[18],
                }

                all_data.append(row)

            except Exception as e:
                print("Parse Error:", e)

        time.sleep(1)

    except Exception as e:

        print("Request Error:", e)


# Converting to Data Frame
df = pd.DataFrame(all_data)


# Storing it in a CSV
csv_file = "NSE Stock data.csv"

df.to_csv(csv_file, index=False)
print(df.head())


  Exchange      Symbol                      Company    Price  Change %  \
0      NSE    RELIANCE  Reliance Industries Limited  1364.00 -1.743265   
1      NSE    HDFCBANK            HDFC Bank Limited   750.45 -1.728541   
2      NSE  BHARTIARTL        Bharti Airtel Limited  1756.80 -0.170474   
3      NSE   ICICIBANK           ICICI Bank Limited  1240.30 -2.060960   
4      NSE        SBIN          State Bank of India   974.60  0.102712   

     Volume  Relative Volume    Market Cap        P/E       P/B        ROE  \
0  24357500         1.046539  1.878519e+13  22.851514       NaN   9.269837   
1  43937132         1.234171  1.177063e+13  15.231378       NaN  13.765023   
2  11945399         1.304434  1.070700e+13  35.205496  8.575268  26.624249   
3  21110708         1.181233  9.076432e+12  16.585986       NaN  16.014888   
4  25787092         1.311626  8.986007e+12  10.706455       NaN  15.380589   

        ROA  EPS TTM  EPS Growth YoY  Dividend Yield        RSI      SMA20  \
0  3.913